# Machine Learning básico con el dataset Iris

Este notebook enseña un flujo básico de Machine Learning usando el famoso dataset **Iris**, que contiene medidas de flores:

- `sepal length`: largo del sépalo
- `sepal width`: ancho del sépalo
- `petal length`: largo del pétalo
- `petal width`: ancho del pétalo

El objetivo es entrenar un modelo para clasificar una flor en una de estas especies:

- Setosa
- Versicolor
- Virginica

Además, el notebook guarda el modelo entrenado en un archivo llamado `modelo_iris.pkl`.


## 1. Importar librerías

Usaremos librerías muy conocidas en Python:

- `pandas` para trabajar con tablas de datos.
- `matplotlib` para gráficos.
- `sklearn` para Machine Learning.
- `joblib` para guardar el modelo entrenado.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


## 2. Cargar el dataset Iris

El dataset Iris ya viene incluido en `scikit-learn`, por eso no necesitamos descargar ningún archivo externo.


In [ ]:
# Cargar el dataset Iris
iris = load_iris()

# Crear un DataFrame con las columnas del dataset
df = pd.DataFrame(iris.data, columns=iris.feature_names)

# Agregar la columna objetivo numérica
df['target'] = iris.target

# Agregar la especie como texto para que sea más fácil de entender
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Mostrar las primeras filas
df.head()


## 3. Explorar los datos

Antes de entrenar un modelo, debemos entender cómo están organizados los datos.


In [ ]:
# Tamaño del dataset: filas y columnas
df.shape


In [ ]:
# Información general del dataset
df.info()


In [ ]:
# Estadísticas básicas
df.describe()


In [ ]:
# Cantidad de registros por especie
df['species'].value_counts()


## 4. Gráfico básico de los datos

Vamos a visualizar la relación entre el largo y ancho del pétalo. Esta relación suele separar bastante bien las especies.


In [ ]:
plt.figure(figsize=(8, 5))

for especie in df['species'].unique():
    datos = df[df['species'] == especie]
    plt.scatter(
        datos['petal length (cm)'],
        datos['petal width (cm)'],
        label=especie
    )

plt.title('Dataset Iris: largo y ancho del pétalo')
plt.xlabel('Petal length (cm)')
plt.ylabel('Petal width (cm)')
plt.legend()
plt.grid(True)
plt.show()


## 5. Separar variables de entrada y variable objetivo

En Machine Learning normalmente dividimos los datos en:

- `X`: variables de entrada o características.
- `y`: variable objetivo o clase que queremos predecir.


In [ ]:
# Variables de entrada: medidas de la flor
X = df[iris.feature_names]

# Variable objetivo: especie en formato numérico
y = df['target']

print('Variables de entrada:')
print(X.head())

print('
Variable objetivo:')
print(y.head())


## 6. Dividir datos en entrenamiento y prueba

Dividimos el dataset en dos partes:

- Datos de entrenamiento: sirven para enseñar al modelo.
- Datos de prueba: sirven para evaluar si el modelo aprendió bien.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print('Tamaño de X_train:', X_train.shape)
print('Tamaño de X_test:', X_test.shape)
print('Tamaño de y_train:', y_train.shape)
print('Tamaño de y_test:', y_test.shape)


## 7. Crear y entrenar el modelo

Usaremos un modelo sencillo llamado **Árbol de Decisión**.

Este modelo aprende reglas del tipo:

> Si el largo del pétalo es menor que cierto valor, entonces probablemente es Setosa.


In [ ]:
# Crear el modelo
modelo = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Entrenar el modelo con los datos de entrenamiento
modelo.fit(X_train, y_train)

print('Modelo entrenado correctamente')


## 8. Evaluar el modelo

Ahora usamos los datos de prueba para medir qué tan bien predice el modelo.


In [ ]:
# Hacer predicciones con los datos de prueba
y_pred = modelo.predict(X_test)

# Calcular exactitud del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy del modelo: {accuracy:.2f}')


In [ ]:
# Reporte de clasificación
print(classification_report(y_test, y_pred, target_names=iris.target_names))


## 9. Matriz de confusión

La matriz de confusión permite ver cuántas flores fueron clasificadas correctamente y cuántas fueron confundidas con otra especie.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
)

disp.plot()
plt.title('Matriz de confusión - Modelo Iris')
plt.show()


## 10. Visualizar el árbol de decisión

Este gráfico permite ver las reglas que aprendió el modelo.


In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    modelo,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True
)
plt.title('Árbol de decisión entrenado con Iris')
plt.show()


## 11. Hacer una predicción nueva

Ahora vamos a ingresar manualmente las medidas de una flor para que el modelo prediga su especie.

El orden de las variables es:

1. Sepal length
2. Sepal width
3. Petal length
4. Petal width


In [ ]:
# Nueva flor de ejemplo
nueva_flor = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=iris.feature_names
)

# Predecir especie
prediccion = modelo.predict(nueva_flor)

# Convertir el resultado numérico a nombre de especie
especie_predicha = iris.target_names[prediccion[0]]

print('Datos de la nueva flor:')
print(nueva_flor)
print('
Especie predicha:', especie_predicha)


## 12. Guardar el modelo entrenado

Guardaremos el modelo en un archivo llamado `modelo_iris.pkl`.

Este archivo puede cargarse luego en otro programa para hacer predicciones sin volver a entrenar el modelo.


In [ ]:
# Guardar el modelo entrenado
joblib.dump(modelo, 'modelo_iris.pkl')

print('Modelo guardado como modelo_iris.pkl')


## 13. Cargar el modelo guardado y probarlo

Esto permite comprobar que el archivo del modelo se grabó correctamente.


In [ ]:
# Cargar el modelo desde el archivo
modelo_cargado = joblib.load('modelo_iris.pkl')

# Usar el modelo cargado para predecir nuevamente
prediccion_cargada = modelo_cargado.predict(nueva_flor)
especie_cargada = iris.target_names[prediccion_cargada[0]]

print('Predicción usando el modelo cargado:', especie_cargada)


## 14. Conclusión

En este notebook aprendiste el flujo básico de Machine Learning:

1. Cargar datos.
2. Explorar datos.
3. Separar variables de entrada y objetivo.
4. Dividir datos en entrenamiento y prueba.
5. Entrenar un modelo.
6. Evaluarlo.
7. Hacer nuevas predicciones.
8. Guardar el modelo entrenado.

Este mismo flujo se puede aplicar después a datasets de ciberseguridad, logs, fraudes o eventos sospechosos.
